# Phase 2b — Multi-Regulation Chunking

Extends Phase 2 to produce a single unified chunk file covering all three regulations.

**What this notebook does:**
1. Loads existing DORA chunks (`dora_chunks.json`) — no re-chunking
2. Patches `regulation='dora'` and `jurisdiction='eu'` onto every DORA chunk
3. Loads new NIS2, PS21/3, and cross-regulation documents from `multi_reg_manifest.json`
4. Chunks new documents using the same strategy as Phase 2:
   - Article-aware chunking for EU documents (Article N pattern)
   - Section-aware chunking for UK PS21/3 documents (Chapter/paragraph N.N pattern)
   - Sliding window fallback for documents with no section map
5. Combines DORA + new chunks into `all_chunks.json`
6. Saves a unified index for Phase 3

**Prerequisites:** Phase 1 + Phase 1b complete.
Required files:
- `dora_rag/dora_chunks/dora_chunks.json`  (from Phase 2)
- `dora_rag/dora_cleaned/manifest.json`     (from Phase 1)
- `dora_rag/dora_cleaned/multi_reg_manifest.json` (from Phase 1b)

In [1]:
# Cell 1 — Imports and configuration
import re
import json
import statistics
from pathlib import Path
from dataclasses import dataclass, asdict, field
from typing import List, Optional
from collections import Counter

BASE_DIR    = Path('./dora_rag')
CLEANED_DIR = BASE_DIR / 'dora_cleaned'
CHUNKS_DIR  = BASE_DIR / 'dora_chunks'
CHUNKS_DIR.mkdir(parents=True, exist_ok=True)

# Same chunking parameters as Phase 2
MAX_CHUNK_TOKENS = 512
OVERLAP_TOKENS   = 64
CHARS_PER_TOKEN  = 4
MAX_CHUNK_CHARS  = MAX_CHUNK_TOKENS * CHARS_PER_TOKEN   # ~2048
OVERLAP_CHARS    = OVERLAP_TOKENS   * CHARS_PER_TOKEN   # ~256

print('Phase 2b — Multi-Regulation Chunking')
print(f'  Max chunk : ~{MAX_CHUNK_TOKENS} tokens (~{MAX_CHUNK_CHARS} chars)')
print(f'  Overlap   : ~{OVERLAP_TOKENS} tokens (~{OVERLAP_CHARS} chars)')
print()

# Verify required input files exist
DORA_CHUNKS_FILE     = CHUNKS_DIR  / 'dora_chunks.json'
DORA_MANIFEST        = CLEANED_DIR / 'manifest.json'
MULTI_REG_MANIFEST   = CLEANED_DIR / 'multi_reg_manifest.json'

for f in [DORA_CHUNKS_FILE, DORA_MANIFEST, MULTI_REG_MANIFEST]:
    status = '✓' if f.exists() else '✗ MISSING'
    print(f'  {status}  {f}')

missing = [f for f in [DORA_CHUNKS_FILE, DORA_MANIFEST, MULTI_REG_MANIFEST] if not f.exists()]
if missing:
    raise FileNotFoundError(
        f'{len(missing)} required file(s) missing.\n'
        'Run Phase 1, Phase 1b, and Phase 2 first.'
    )
print('\n✓ All input files present')


Phase 2b — Multi-Regulation Chunking
  Max chunk : ~512 tokens (~2048 chars)
  Overlap   : ~64 tokens (~256 chars)

  ✓  dora_rag/dora_chunks/dora_chunks.json
  ✓  dora_rag/dora_cleaned/manifest.json
  ✓  dora_rag/dora_cleaned/multi_reg_manifest.json

✓ All input files present


In [2]:
# Cell 2 — Extended DoraChunk dataclass
#
# Adds four new fields to the original Phase 2 dataclass:
#   regulation   : which regulatory framework ("dora" | "nis2" | "ps21_3" | "cross_reg")
#   jurisdiction : legal jurisdiction       ("eu" | "uk" | "eu_uk")
#   level        : regulatory level         (1 | 2 | 3)  — was missing from Phase 2 original
#   section_ref  : PS21/3 paragraph ref     ("5.1" | "Chapter 3" | None)
#
# All existing DORA fields are preserved exactly — downstream code is unaffected.

@dataclass
class DoraChunk:
    # ── Identity ──────────────────────────────────────────────────────────────
    chunk_id:         str
    text:             str
    text_with_header: str
    # ── Document metadata ─────────────────────────────────────────────────────
    source_doc:       str
    doc_type:         str
    regulation:       str           # NEW: "dora" | "nis2" | "ps21_3" | "cross_reg"
    jurisdiction:     str           # NEW: "eu" | "uk" | "eu_uk"
    level:            int           # NEW (was missing in Phase 2): 1 | 2 | 3
    # ── Location within document ──────────────────────────────────────────────
    article_num:      Optional[int] # EU documents: Article N
    article_title:    str
    section_ref:      Optional[str] # NEW: PS21/3 paragraph refs ("5.1", "Chapter 3")
    paragraph_idx:    int
    chapter:          Optional[str]
    # ── Semantic metadata ─────────────────────────────────────────────────────
    topic_tags:       List[str]
    entity_scope:     List[str]
    # ── Position ──────────────────────────────────────────────────────────────
    char_start:       int
    char_end:         int
    token_estimate:   int


def estimate_tokens(text: str) -> int:
    return len(text) // 4


print('✓ DoraChunk dataclass defined')
print(f'  Fields: {[f for f in DoraChunk.__dataclass_fields__]}')
print(f'  New fields: regulation, jurisdiction, level, section_ref')


✓ DoraChunk dataclass defined
  Fields: ['chunk_id', 'text', 'text_with_header', 'source_doc', 'doc_type', 'regulation', 'jurisdiction', 'level', 'article_num', 'article_title', 'section_ref', 'paragraph_idx', 'chapter', 'topic_tags', 'entity_scope', 'char_start', 'char_end', 'token_estimate']
  New fields: regulation, jurisdiction, level, section_ref


In [3]:
# Cell 3 — Topic and entity tagging
#
# Extends Phase 2 keyword maps with NIS2 and PS21/3 specific terms.
# DORA-specific keywords are preserved unchanged.

TOPIC_KEYWORDS = {
    # DORA topics (unchanged from Phase 2)
    'ict_risk_management':    ['ict risk', 'information and communication', 'risk management framework',
                                'risk appetite', 'risk tolerance'],
    'incident_reporting':     ['incident', 'major ict', 'reporting', 'notification', 'classify'],
    'third_party_risk':       ['third-party', 'third party', 'ict third', 'outsourc', 'sub-outsourc',
                                'service provider', 'ctpp', 'critical third'],
    'resilience_testing':     ['resilience testing', 'tlpt', 'penetration test', 'vulnerability',
                                'tiber', 'threat-led'],
    'governance':             ['management body', 'board', 'governance', 'oversight', 'accountability',
                                'responsibility'],
    'business_continuity':    ['business continuity', 'disaster recovery', 'backup', 'crisis',
                                'continuity plan'],
    'information_sharing':    ['information sharing', 'threat intelligence', 'cyber threat', 'share'],
    'data_security':          ['data security', 'encryption', 'access control', 'authentication',
                                'data integrity', 'confidentiality'],
    'supervision':            ['competent authorit', 'supervisor', 'eba', 'eiopa', 'esma',
                                'oversight framework', 'lead overseer'],
    # NIS2 specific topics
    'network_security':       ['network security', 'information system', 'nis2', 'network and information',
                                'essential entit', 'important entit'],
    'supply_chain_security':  ['supply chain', 'supplier', 'vendor risk', 'procurement security'],
    'cyber_hygiene':          ['cyber hygiene', 'patch management', 'asset management', 'vulnerability management'],
    # PS21/3 specific topics
    'impact_tolerance':       ['impact tolerance', 'important business service', 'mapping', 'scenario testing'],
    'operational_resilience': ['operational resilience', 'self-assessment', 'stay within', 'disruption'],
    'critical_third_parties': ['critical third part', 'ctp', 'systemic', 'financial stability'],
}

ENTITY_KEYWORDS = {
    # DORA entities (unchanged from Phase 2)
    'credit_institutions':    ['credit institution', 'bank'],
    'investment_firms':       ['investment firm'],
    'insurance':              ['insurance', 'reinsurance', 'insurer'],
    'payment_institutions':   ['payment institution', 'e-money', 'electronic money'],
    'crypto_assets':          ['crypto-asset', 'crypto asset', 'casp'],
    'ict_third_party':        ['ict third-party provider', 'ctpp', 'critical third-party'],
    'simplified_framework':   ['simplified ict risk', 'small and non-interconnected'],
    # NIS2 entities
    'essential_entities':     ['essential entit', 'operators of essential services'],
    'important_entities':     ['important entit'],
    'dns_providers':          ['dns service provider', 'domain name'],
    # PS21/3 entities
    'uk_regulated_firms':     ['fca-regulated', 'pra-regulated', 'authorised firm', 'uk firm'],
    'systemic_firms':         ['systemic', 'too big to fail', 'g-sib', 'g-sii'],
}


def infer_topic_tags(text: str) -> List[str]:
    t = text.lower()
    return [topic for topic, kws in TOPIC_KEYWORDS.items() if any(kw in t for kw in kws)]


def infer_entity_scope(text: str) -> List[str]:
    t = text.lower()
    return [entity for entity, kws in ENTITY_KEYWORDS.items() if any(kw in t for kw in kws)]


print('✓ Topic and entity taggers ready')
print(f'  Topics   : {len(TOPIC_KEYWORDS)} categories ({len(TOPIC_KEYWORDS)-9} new vs Phase 2)')
print(f'  Entities : {len(ENTITY_KEYWORDS)} categories ({len(ENTITY_KEYWORDS)-7} new vs Phase 2)')


✓ Topic and entity taggers ready
  Topics   : 15 categories (6 new vs Phase 2)
  Entities : 12 categories (5 new vs Phase 2)


In [4]:
# Cell 4 — Chunking functions
#
# Three chunkers to handle different document structures:
#   chunk_article()  — EU documents: Article N header format (same as Phase 2)
#   chunk_section()  — PS21/3 UK documents: Chapter/paragraph header format
#   sliding_window() — fallback for documents with no section map
#
# All three produce DoraChunk objects with the full extended metadata schema.

def split_into_paragraphs(text: str) -> List[str]:
    paragraphs = re.split(r'\n{2,}', text.strip())
    return [p.strip() for p in paragraphs if len(p.strip()) > 40]


def make_chunk_id(doc_name: str, article_num: Optional[int],
                  para_idx: int, section_ref: Optional[str] = None) -> str:
    doc_short = doc_name[:20].replace(' ', '_')
    if article_num is not None:
        loc_part = f'art{article_num:03d}'
    elif section_ref:
        # Clean section_ref for use in ID: '5.1' → 'sec5_1', 'Chapter 3' → 'ch3'
        clean = re.sub(r'[^\w]', '_', section_ref.lower())
        loc_part = f'sec_{clean}'
    else:
        loc_part = 'preamble'
    return f'{doc_short}__{loc_part}__p{para_idx:02d}'


def _build_chunks_from_paragraphs(
    text: str, header: str, article_num: Optional[int], article_title: str,
    section_ref: Optional[str], source_doc: str, doc_type: str,
    regulation: str, jurisdiction: str, level: int, char_start: int
) -> List[DoraChunk]:
    """
    Shared paragraph-splitting logic used by both chunk_article() and chunk_section().
    Short text → 1 chunk. Long text → paragraph-boundary chunks with overlap.
    """
    chunks = []

    def make(chunk_text: str, idx: int) -> DoraChunk:
        return DoraChunk(
            chunk_id         = make_chunk_id(source_doc, article_num, idx, section_ref),
            text             = chunk_text,
            text_with_header = header + chunk_text,
            source_doc       = source_doc,
            doc_type         = doc_type,
            regulation       = regulation,
            jurisdiction     = jurisdiction,
            level            = level,
            article_num      = article_num,
            article_title    = article_title,
            section_ref      = section_ref,
            paragraph_idx    = idx,
            chapter          = None,
            topic_tags       = infer_topic_tags(chunk_text),
            entity_scope     = infer_entity_scope(chunk_text),
            char_start       = char_start,
            char_end         = char_start + len(chunk_text),
            token_estimate   = estimate_tokens(chunk_text),
        )

    # Short text: single chunk
    if len(text) <= MAX_CHUNK_CHARS:
        return [make(text, 0)]

    # Long text: paragraph-boundary split with overlap
    paragraphs = split_into_paragraphs(text)
    current_paras, current_chars, chunk_idx = [], 0, 0

    for para in paragraphs:
        if current_chars + len(para) > MAX_CHUNK_CHARS and current_paras:
            chunks.append(make('\n\n'.join(current_paras), chunk_idx))
            chunk_idx += 1
            # Overlap: keep last paragraph(s)
            overlap_paras, overlap_chars = [], 0
            for p in reversed(current_paras):
                if overlap_chars + len(p) <= OVERLAP_CHARS:
                    overlap_paras.insert(0, p)
                    overlap_chars += len(p)
                else:
                    break
            current_paras, current_chars = overlap_paras, overlap_chars
        current_paras.append(para)
        current_chars += len(para)

    if current_paras:
        chunks.append(make('\n\n'.join(current_paras), chunk_idx))

    return chunks


def chunk_article(article_text, article_num, article_title,
                  source_doc, doc_type, regulation, jurisdiction, level, char_start):
    """Chunk an EU-style Article N section. Identical strategy to Phase 2."""
    header = f'Article {article_num} — {article_title}\n\n'
    return _build_chunks_from_paragraphs(
        article_text, header, article_num, article_title,
        None, source_doc, doc_type, regulation, jurisdiction, level, char_start
    )


def chunk_section(section_text, section_ref, section_title,
                  source_doc, doc_type, regulation, jurisdiction, level, char_start):
    """
    Chunk a PS21/3-style section (Chapter N or paragraph N.N).
    Uses section_ref instead of article_num in the header and chunk ID.
    """
    header = f'{section_ref} — {section_title}\n\n'
    return _build_chunks_from_paragraphs(
        section_text, header, None, section_title,
        section_ref, source_doc, doc_type, regulation, jurisdiction, level, char_start
    )


def sliding_window_chunks(text, source_doc, doc_type,
                           regulation, jurisdiction, level):
    """Sliding window fallback for documents with no section map."""
    chunks, start, idx = [], 0, 0
    while start < len(text):
        end        = min(start + MAX_CHUNK_CHARS, len(text))
        chunk_text = text[start:end]
        chunks.append(DoraChunk(
            chunk_id         = make_chunk_id(source_doc, None, idx),
            text             = chunk_text,
            text_with_header = chunk_text,
            source_doc       = source_doc,
            doc_type         = doc_type,
            regulation       = regulation,
            jurisdiction     = jurisdiction,
            level            = level,
            article_num      = None,
            article_title    = 'Unknown',
            section_ref      = None,
            paragraph_idx    = idx,
            chapter          = None,
            topic_tags       = infer_topic_tags(chunk_text),
            entity_scope     = infer_entity_scope(chunk_text),
            char_start       = start,
            char_end         = end,
            token_estimate   = estimate_tokens(chunk_text),
        ))
        start += MAX_CHUNK_CHARS - OVERLAP_CHARS
        idx   += 1
    return chunks


print('✓ Chunking functions ready')
print('  chunk_article()         — EU documents (Article N pattern)')
print('  chunk_section()         — PS21/3 UK documents (Chapter/para N.N pattern)')
print('  sliding_window_chunks() — fallback for documents with no section map')


✓ Chunking functions ready
  chunk_article()         — EU documents (Article N pattern)
  chunk_section()         — PS21/3 UK documents (Chapter/para N.N pattern)
  sliding_window_chunks() — fallback for documents with no section map


In [5]:
# Cell 5 — Load and patch existing DORA chunks
#
# Loads dora_chunks.json from Phase 2 and adds the three new fields:
#   regulation   = "dora"    (all existing DORA chunks)
#   jurisdiction = "eu"      (all existing DORA chunks)
#   level        = from manifest (already fixed by 02b_patch_level_metadata.ipynb)
#   section_ref  = None      (DORA uses article_num, not section_ref)
#
# The existing chunks are NOT re-chunked — they are reused as-is.

# Build level lookup from Phase 1 manifest
dora_manifest  = json.loads(DORA_MANIFEST.read_text())
level_lookup   = {e['name']: e['level'] for e in dora_manifest}

# Load existing DORA chunks
raw_dora = json.loads(DORA_CHUNKS_FILE.read_text(encoding='utf-8'))

dora_chunks: List[DoraChunk] = []
patched = 0

for c in raw_dora:
    level = c.get('level', 0)
    if level == 0:
        level = level_lookup.get(c.get('source_doc', ''), 0)
        patched += 1

    dora_chunks.append(DoraChunk(
        chunk_id         = c['chunk_id'],
        text             = c['text'],
        text_with_header = c['text_with_header'],
        source_doc       = c['source_doc'],
        doc_type         = c['doc_type'],
        regulation       = 'dora',     # patch
        jurisdiction     = 'eu',       # patch
        level            = level,
        article_num      = c.get('article_num'),
        article_title    = c.get('article_title', ''),
        section_ref      = None,       # DORA uses article_num
        paragraph_idx    = c.get('paragraph_idx', 0),
        chapter          = c.get('chapter'),
        topic_tags       = c.get('topic_tags', []),
        entity_scope     = c.get('entity_scope', []),
        char_start       = c.get('char_start', 0),
        char_end         = c.get('char_end', 0),
        token_estimate   = c.get('token_estimate', 0),
    ))

print(f'✓ Loaded {len(dora_chunks)} DORA chunks from Phase 2')
print(f'  Patched regulation/jurisdiction: {len(dora_chunks)} chunks → regulation=dora, jurisdiction=eu')
print(f'  Patched level from manifest    : {patched} chunks')

by_level = Counter(c.level for c in dora_chunks)
print(f'  Level distribution: {dict(sorted(by_level.items()))}')


✓ Loaded 390 DORA chunks from Phase 2
  Patched regulation/jurisdiction: 390 chunks → regulation=dora, jurisdiction=eu
  Patched level from manifest    : 0 chunks
  Level distribution: {1: 132, 2: 248, 3: 10}


In [6]:
# Cell 6 — Load new documents from multi-regulation manifest

multi_manifest = json.loads(MULTI_REG_MANIFEST.read_text())

new_docs = {}
for entry in multi_manifest:
    name     = entry['name']
    txt_path = Path(entry['txt_file'])

    if not txt_path.exists():
        print(f'  ✗ Skipping {name} — text file not found: {txt_path}')
        continue

    cleaned_text = txt_path.read_text(encoding='utf-8')

    article_map = []
    if entry.get('article_map_file'):
        map_path = Path(entry['article_map_file'])
        if map_path.exists():
            article_map = json.loads(map_path.read_text())

    new_docs[name] = {
        **entry,
        'cleaned_text': cleaned_text,
        'article_map':  article_map,
    }

    reg   = entry['regulation']
    jur   = entry['jurisdiction']
    sects = len(article_map)
    print(f'  ✓ {name} [{reg}/{jur}] — {len(cleaned_text):,} chars, {sects} sections')

print(f'\n✓ Loaded {len(new_docs)}/{len(multi_manifest)} new documents')

by_reg = Counter(d['regulation'] for d in new_docs.values())
for reg, cnt in sorted(by_reg.items()):
    print(f'  {reg:<12}: {cnt} documents')


  ✓ NIS2_Directive_2022_2555 [nis2/eu] — 276,397 chars, 56 sections
  ✓ NIS2_CER_Directive_2022_2557 [nis2/eu] — 126,862 chars, 29 sections
  ✓ NIS2_IR_2024_2690_Cybersecurity_Requirements [nis2/eu] — 116,912 chars, 16 sections
  ✓ NIS2_ENISA_Technical_Implementation_Guidance_2024_2690 [nis2/eu] — 397,965 chars, 2 sections
  ✓ PS21_3_FCA_PRA_Operational_Resilience [ps21_3/uk] — 165,843 chars, 8 sections
  ✓ SS1_21_PRA_Operational_Resilience [ps21_3/uk] — 48,086 chars, 2 sections
  ✓ FCA_FG22_5_Operational_Resilience [ps21_3/uk] — 334,240 chars, 25 sections
  ✓ DP3_22_Critical_Third_Parties_UK [ps21_3/uk] — 175,692 chars, 4 sections
  ✓ EC_Guidelines_DORA_NIS2_Relationship_2023 [cross_reg/eu] — 32,585 chars, 1 sections
  ✓ PS16_24_PRA_PS24_16_FCA_Critical_Third_Parties [cross_reg/uk] — 174,034 chars, 10 sections
  ✓ DORA_ESA_QA_Implementation_Guidance [cross_reg/eu] — 76,512 chars, 0 sections
  ✓ DORA_ESA_Guidelines_Simplified_ICT_Framework [cross_reg/eu] — 268,201 chars, 18 sections

✓

In [7]:
# Cell 7 — Chunk all new documents
#
# Routing logic:
#   jurisdiction == "uk"  → chunk_section()  (PS21/3 Chapter/paragraph structure)
#   jurisdiction == "eu" or "eu_uk" with article_map → chunk_article()  (Article N)
#   no article_map → sliding_window_chunks() (fallback)

new_chunks: List[DoraChunk] = []

for doc_name, doc in new_docs.items():
    text        = doc['cleaned_text']
    article_map = doc['article_map']
    doc_type    = doc['doc_type']
    regulation  = doc['regulation']
    jurisdiction= doc['jurisdiction']
    level       = doc['level']
    doc_chunks  = []

    if not article_map:
        # No section map — sliding window fallback
        print(f'  {doc_name}: no section map → sliding window')
        doc_chunks = sliding_window_chunks(
            text, doc_name, doc_type, regulation, jurisdiction, level
        )

    elif jurisdiction == 'uk':
        # PS21/3 UK documents — section-aware chunking
        print(f'  {doc_name}: PS21/3 structure → section chunker')
        for section in article_map:
            sec_text  = text[section['char_start']:section['char_end']]
            sec_ref   = section.get('section_ref', '')
            sec_title = section.get('title', '')
            doc_chunks.extend(chunk_section(
                section_text = sec_text,
                section_ref  = sec_ref,
                section_title= sec_title,
                source_doc   = doc_name,
                doc_type     = doc_type,
                regulation   = regulation,
                jurisdiction = jurisdiction,
                level        = level,
                char_start   = section['char_start'],
            ))

    else:
        # EU documents (NIS2, cross-reg) — article-aware chunking
        print(f'  {doc_name}: EU structure → article chunker')
        for article in article_map:
            art_text  = text[article['char_start']:article['char_end']]
            doc_chunks.extend(chunk_article(
                article_text  = art_text,
                article_num   = article['article_num'],
                article_title = article['title'],
                source_doc    = doc_name,
                doc_type      = doc_type,
                regulation    = regulation,
                jurisdiction  = jurisdiction,
                level         = level,
                char_start    = article['char_start'],
            ))

    new_chunks.extend(doc_chunks)
    avg_tok = sum(c.token_estimate for c in doc_chunks) / len(doc_chunks) if doc_chunks else 0
    print(f'    → {len(doc_chunks)} chunks (avg ~{avg_tok:.0f} tokens)')

print(f'\n✓ Chunked {len(new_docs)} new documents → {len(new_chunks)} new chunks')

by_reg_new = Counter(c.regulation for c in new_chunks)
for reg, cnt in sorted(by_reg_new.items()):
    print(f'  {reg:<12}: {cnt} chunks')


  NIS2_Directive_2022_2555: EU structure → article chunker
    → 90 chunks (avg ~395 tokens)
  NIS2_CER_Directive_2022_2557: EU structure → article chunker
    → 45 chunks (avg ~413 tokens)
  NIS2_IR_2024_2690_Cybersecurity_Requirements: EU structure → article chunker
    → 37 chunks (avg ~563 tokens)
  NIS2_ENISA_Technical_Implementation_Guidance_2024_2690: EU structure → article chunker
    → 17 chunks (avg ~5758 tokens)
  PS21_3_FCA_PRA_Operational_Resilience: PS21/3 structure → section chunker
    → 21 chunks (avg ~1895 tokens)
  SS1_21_PRA_Operational_Resilience: PS21/3 structure → section chunker
    → 16 chunks (avg ~731 tokens)
  FCA_FG22_5_Operational_Resilience: PS21/3 structure → section chunker
    → 25 chunks (avg ~3327 tokens)
  DP3_22_Critical_Third_Parties_UK: PS21/3 structure → section chunker
    → 57 chunks (avg ~599 tokens)
  EC_Guidelines_DORA_NIS2_Relationship_2023: EU structure → article chunker
    → 3 chunks (avg ~549 tokens)
  PS16_24_PRA_PS24_16_FCA_Critical_

In [8]:
# Cell 8 — Combine DORA + new chunks into unified corpus

all_chunks: List[DoraChunk] = dora_chunks + new_chunks

print('COMBINED CORPUS')
print('=' * 55)
print(f'  DORA chunks    : {len(dora_chunks)}')
print(f'  New chunks     : {len(new_chunks)}')
print(f'  Total          : {len(all_chunks)}')
print()

# By regulation
by_reg = Counter(c.regulation for c in all_chunks)
print('  By regulation:')
for reg, cnt in sorted(by_reg.items()):
    pct = 100 * cnt // len(all_chunks)
    bar = '█' * (cnt * 30 // len(all_chunks))
    print(f'    {reg:<12}: {cnt:>4}  {bar} ({pct}%)')

# By level
by_level = Counter(c.level for c in all_chunks)
print('\n  By regulatory level:')
for lv, cnt in sorted(by_level.items()):
    print(f'    Level {lv}      : {cnt} chunks')

# By jurisdiction
by_jur = Counter(c.jurisdiction for c in all_chunks)
print('\n  By jurisdiction:')
for jur, cnt in sorted(by_jur.items()):
    print(f'    {jur:<10}   : {cnt} chunks')

# Check for chunk_id uniqueness
all_ids = [c.chunk_id for c in all_chunks]
dupes   = len(all_ids) - len(set(all_ids))
print(f'\n  Chunk ID duplicates: {dupes}', '✓' if dupes == 0 else '⚠ RESOLVE BEFORE INDEXING')


COMBINED CORPUS
  DORA chunks    : 390
  New chunks     : 436
  Total          : 826

  By regulation:
    cross_reg   :  128  ████ (15%)
    dora        :  390  ██████████████ (47%)
    nis2        :  189  ██████ (22%)
    ps21_3      :  119  ████ (14%)

  By regulatory level:
    Level 1      : 288 chunks
    Level 2      : 447 chunks
    Level 3      : 91 chunks

  By jurisdiction:
    eu           : 643 chunks
    uk           : 183 chunks

  Chunk ID duplicates: 28 ⚠ RESOLVE BEFORE INDEXING


In [9]:
# Cell 9 — Quality analysis

token_sizes = [c.token_estimate for c in all_chunks]

print('CHUNK SIZE DISTRIBUTION (token estimates)')
print('=' * 55)
print(f'  Total chunks  : {len(token_sizes)}')
print(f'  Min tokens    : {min(token_sizes)}')
print(f'  Max tokens    : {max(token_sizes)}')
print(f'  Mean tokens   : {statistics.mean(token_sizes):.0f}')
print(f'  Median tokens : {statistics.median(token_sizes):.0f}')
print(f'  Stdev tokens  : {statistics.stdev(token_sizes):.0f}')

buckets = [(0,100),(100,200),(200,300),(300,400),(400,512),(512,99999)]
print('\n  Bucket distribution:')
for lo, hi in buckets:
    count = sum(1 for t in token_sizes if lo <= t < hi)
    bar   = '█' * (count * 40 // max(1, len(token_sizes)))
    label = f'{lo:>4}–{hi if hi < 9999 else "∞":>4} tokens'
    print(f'    {label}: {bar} {count}')

# Metadata coverage
print('\n  METADATA COVERAGE')
print('  ' + '-' * 45)
with_art  = sum(1 for c in all_chunks if c.article_num is not None)
with_sec  = sum(1 for c in all_chunks if c.section_ref is not None)
with_top  = sum(1 for c in all_chunks if c.topic_tags)
with_ent  = sum(1 for c in all_chunks if c.entity_scope)
with_reg  = sum(1 for c in all_chunks if c.regulation)
n         = len(all_chunks)
print(f'  regulation   : {with_reg}/{n} ({100*with_reg//n}%)')
print(f'  article_num  : {with_art}/{n} ({100*with_art//n}%) — EU documents')
print(f'  section_ref  : {with_sec}/{n} ({100*with_sec//n}%) — PS21/3 documents')
print(f'  topic_tags   : {with_top}/{n} ({100*with_top//n}%)')
print(f'  entity_scope : {with_ent}/{n} ({100*with_ent//n}%)')


CHUNK SIZE DISTRIBUTION (token estimates)
  Total chunks  : 826
  Min tokens    : 8
  Max tokens    : 50570
  Mean tokens   : 781
  Median tokens : 512
  Stdev tokens  : 2350

  Bucket distribution:
       0– 100 tokens: ██ 56
     100– 200 tokens: ███ 78
     200– 300 tokens: ████ 88
     300– 400 tokens: ███ 74
     400– 512 tokens: ████ 88
     512–   ∞ tokens: █████████████████████ 442

  METADATA COVERAGE
  ---------------------------------------------
  regulation   : 826/826 (100%)
  article_num  : 572/826 (69%) — EU documents
  section_ref  : 183/826 (22%) — PS21/3 documents
  topic_tags   : 738/826 (89%)
  entity_scope : 314/826 (38%)


In [10]:
# Cell 10 — Save unified chunks to disk
#
# Saves three files:
#   all_chunks.json       — full unified corpus (used by Phase 3b for indexing)
#   all_chunks_index.json — lightweight metadata index (no text)
#   dora_chunks.json      — DORA-only chunks with new fields patched in (overwrites Phase 2 output)
#
# Phase 3 (existing DORA-only indexing) continues to work from dora_chunks.json.
# Phase 3b (multi-regulation indexing) uses all_chunks.json.

import shutil

chunks_data = [asdict(c) for c in all_chunks]
dora_data   = [asdict(c) for c in dora_chunks]

# Save unified corpus
all_chunks_path = CHUNKS_DIR / 'all_chunks.json'
all_chunks_path.write_text(json.dumps(chunks_data, indent=2, ensure_ascii=False))
print(f'✓ all_chunks.json       : {len(chunks_data)} chunks ({all_chunks_path.stat().st_size/1024/1024:.2f} MB)')

# Save lightweight index
index_data = [
    {
        'chunk_id':      c['chunk_id'],
        'source_doc':    c['source_doc'],
        'regulation':    c['regulation'],
        'jurisdiction':  c['jurisdiction'],
        'doc_type':      c['doc_type'],
        'level':         c['level'],
        'article_num':   c['article_num'],
        'section_ref':   c['section_ref'],
        'article_title': c['article_title'],
        'paragraph_idx': c['paragraph_idx'],
        'topic_tags':    c['topic_tags'],
        'entity_scope':  c['entity_scope'],
        'token_estimate':c['token_estimate'],
    }
    for c in chunks_data
]
index_path = CHUNKS_DIR / 'all_chunks_index.json'
index_path.write_text(json.dumps(index_data, indent=2))
print(f'✓ all_chunks_index.json : {len(index_data)} entries ({index_path.stat().st_size/1024:.0f} KB)')

# Backup and overwrite dora_chunks.json with patched version
dora_backup = CHUNKS_DIR / 'dora_chunks_pre_2b_backup.json'
shutil.copy(CHUNKS_DIR / 'dora_chunks.json', dora_backup)
(CHUNKS_DIR / 'dora_chunks.json').write_text(json.dumps(dora_data, indent=2, ensure_ascii=False))
print(f'✓ dora_chunks.json      : {len(dora_data)} DORA chunks (patched with new fields)')
print(f'  Backup saved          : {dora_backup.name}')


✓ all_chunks.json       : 826 chunks (5.55 MB)
✓ all_chunks_index.json : 826 entries (419 KB)
✓ dora_chunks.json      : 390 DORA chunks (patched with new fields)
  Backup saved          : dora_chunks_pre_2b_backup.json


In [11]:
# Cell 11 — Final summary

print('\n' + '=' * 65)
print('PHASE 2b COMPLETE — MULTI-REGULATION CHUNKING')
print('=' * 65)

print(f"\n{'Document':<48} {'Reg':<10} {'Chunks':>6}")
print('-' * 66)

by_doc = Counter(c.source_doc for c in all_chunks)
by_doc_reg = {c.source_doc: c.regulation for c in all_chunks}
for doc_name, count in by_doc.most_common():
    reg = by_doc_reg[doc_name]
    print(f"{doc_name[:48]:<48} {reg:<10} {count:>6}")

print('-' * 66)
print(f"{'TOTAL':<58} {len(all_chunks):>6}")

print(f"""
Chunk strategy used:
  EU documents (DORA/NIS2)  : article-aware + paragraph splitting
  UK documents (PS21/3)     : section-aware + paragraph splitting
  No section map            : sliding window fallback
  Max tokens                : ~{MAX_CHUNK_TOKENS}
  Overlap                   : ~{OVERLAP_TOKENS} tokens

New metadata fields added:
  regulation    : "dora" | "nis2" | "ps21_3" | "cross_reg"
  jurisdiction  : "eu" | "uk" | "eu_uk"
  level         : 1 | 2 | 3
  section_ref   : PS21/3 paragraph references ("5.1", "Chapter 3")

Output files:
  dora_rag/dora_chunks/all_chunks.json         ← Phase 3b input
  dora_rag/dora_chunks/all_chunks_index.json
  dora_rag/dora_chunks/dora_chunks.json        ← patched (Phase 3 still works)

Next step → open 03b_indexing_multiregulation.ipynb
""")



PHASE 2b COMPLETE — MULTI-REGULATION CHUNKING

Document                                         Reg        Chunks
------------------------------------------------------------------
L1_DORA_Regulation_2022_2554                     dora          116
NIS2_Directive_2022_2555                         nis2           90
PS16_24_PRA_PS24_16_FCA_Critical_Third_Parties   cross_reg      64
L2_RTS_ICT_Risk_Management                       dora           63
DP3_22_Critical_Third_Parties_UK                 ps21_3         57
L2_ITS_ICT_Incident_Reporting                    dora           47
L2_ITS_Register_of_Information                   dora           47
NIS2_CER_Directive_2022_2557                     nis2           45
DORA_ESA_QA_Implementation_Guidance              cross_reg      43
NIS2_IR_2024_2690_Cybersecurity_Requirements     nis2           37
FCA_FG22_5_Operational_Resilience                ps21_3         25
PS21_3_FCA_PRA_Operational_Resilience            ps21_3         21
L2_RTS_TLPT   